# Graduate voter outreach analysis

Upload **`all_english_voters_cleaned.xlsx`** to Google Drive before you start. Addresses in that file are already cleaned (repeated place names removed, title case).

Suggested Drive path:

`MyDrive/electoral_rolls_output/all_english_voters_cleaned.xlsx`

Run every cell from top to bottom. The notebook does not read the PDFs again. It builds one workbook, `outreach_analysis.xlsx`, with:

- **Outreach_List** — active voters to visit, biggest villages first, so one trip reaches more people
- **Village_Plan**, **Taluka_Plan**, **Part_Plan** — where to send teams
- **Qualification_Mix**, **Age_Gender**, **Occupation_Mix** — who those voters are
- **Shifted_Followup** — people marked shifted; look in the new part instead of the old address
- **Data_Quality** — gaps that hide voters (missing age, pin, qualification)

Deleted, struck-out, and duplicate rows stay off the outreach list so the team does not visit the same person twice or knock on a name that is no longer on the roll.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install openpyxl


## 1. Settings

Change `RAW_XLSX` only if you uploaded the cleaned workbook somewhere else.


In [ ]:
import os
RAW_XLSX = '/content/drive/MyDrive/electoral_rolls_output/all_english_voters_cleaned.xlsx'
WORK_DIR = '/content/drive/MyDrive/electoral_rolls_output'
os.makedirs(WORK_DIR, exist_ok=True)


## 2. Load the cleaned voter file


In [ ]:
import pandas as pd

raw = pd.read_excel(RAW_XLSX, dtype=str)
raw = raw.replace({'None': None, 'nan': None, '': None})
for c in ['elector_name', 'relative_name', 'address', 'qualification', 'occupation', 'epic_no', 'gender', 'age']:
    raw[c] = raw[c].fillna('')
print(f'{len(raw):,} rows from {raw["source_file"].nunique():,} PDFs')
print(raw['status'].value_counts().to_string())


## 3. Prepare village, age, qualification, and duplicate flags

The address is split into locality, village, taluka, district, and pincode. The village is the place named just before the taluka, which is the unit a team can walk.


In [ ]:
import re
import pandas as pd

PIN_RE = re.compile(r"\b(\d{6})\b")


def normalise_qualification(q):
    q = re.sub(r"[^a-z]", "", (q or "").lower())
    if "phd" in q or "doctor" in q:
        return "PhD"
    if any(k in q for k in ("bed", "beed", "bead", "dted", "bped", "teacher", "techer")) or q.startswith("med"):
        return "Education (B.Ed/M.Ed)"
    if (any(k in q for k in ("pharm", "paramac", "nurs", "midwif", "mbbs", "bams", "bhms", "bds", "gnm", "ayurved", "dhms"))
            or q.startswith("mbb") or q in {"md", "ms"}):
        return "Medical/Pharma"
    if (any(k in q for k in ("btech", "mtech", "btec", "mtec", "engineer", "civil", "mech", "mach",
                             "comput", "extc"))
            or q in {"be", "me"}
            or (q.startswith("be") and not q.startswith(("bcom", "bba", "bsc", "bca", "bams", "bhms", "bds", "bph", "bped")))):
        return "Engineering"
    if "diploma" in q or "dipal" in q or "polytech" in q or q in {"dce", "msbte"}:
        return "Diploma"
    if "graduat" in q or "univer" in q:
        return "Graduate (unspecified)"
    rules = [
        ("Law", ["llb", "llm", "law"]),
        ("Commerce", ["bcom", "mcom", "commerce", "ca", "bba", "mba"]),
        ("Science", ["bsc", "msc", "science", "bca", "mca"]),
        ("Arts", ["ba", "ma", "arts", "bachelorofarts", "masterofarts"]),
    ]
    for label, keys in rules:
        if any(q == k or q.startswith(k) for k in keys):
            return label
    return "Other/Unclear" if q else "Missing"


def parse_address(addr):
    parts = [p.strip() for p in str(addr or "").split(",") if p.strip()]
    pin = None
    no_pin = []
    for p in parts:
        if PIN_RE.fullmatch(p):
            pin = pin or p
        else:
            no_pin.append(p)
    if pin is None:
        found = PIN_RE.search(str(addr or ""))
        pin = found.group(1) if found else None

    places = []
    for p in no_pin:
        if re.fullmatch(r"[\d/ -]+", p):
            continue
        p = re.sub(r"^(at/?p|at|m/post/?)\s*", "", p, flags=re.I).strip()
        if p:
            places.append(p)
    district = taluka = village = locality = None
    if len(places) == 1:
        village = places[0]
    elif len(places) == 2:
        village, district = places
    else:
        district = places[-1]
        taluka = places[-2]
        village = places[-3]
        if len(places) > 3:
            locality = ", ".join(places[:-3])
    return village, taluka, district, pin, locality


def clean_name(name):
    toks = str(name or "").split()
    if len(toks) > 3 and (toks[-1].lower() == toks[-2].lower()
                          or toks[2].lower() in [t.lower() for t in toks[3:]]):
        return " ".join(toks[:3])
    return str(name or "")


def normalise_occupation(value):
    q = re.sub(r"[^a-z]", "", (value or "").lower())
    if not q:
        return "Missing"
    if "agri" in q or "farm" in q:
        return "Farmer"
    if "teach" in q:
        return "Teacher"
    if "engin" in q:
        return "Engineer"
    if "doct" in q:
        return "Doctor"
    if "nurs" in q:
        return "Nurse"
    if "business" in q:
        return "Business"
    if "gov" in q:
        return "Government service"
    if "police" in q:
        return "Police"
    if q in {"other", "oth", "nil", "na"}:
        return "Other"
    return str(value).strip().title()


def finalise(df):
    df = df.copy()
    df["age"] = pd.to_numeric(df["age"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")
    df["gender"] = df["gender"].astype(str).str.extract(r"([MFT])", flags=re.I)[0].str.upper()
    df["qualification_group"] = df["qualification"].map(normalise_qualification)
    df["has_epic"] = df["epic_no"].fillna("").str.len() > 0
    addr = df["address"].fillna("").apply(parse_address)
    df[["village", "taluka", "addr_district", "pincode", "locality"]] = pd.DataFrame(addr.tolist(), index=df.index)
    for c in ["village", "taluka", "addr_district", "locality"]:
        df[c] = df[c].str.title()
    df["name_clean"] = df["elector_name"].map(clean_name).str.title()

    def key(r):
        t = str(r["name_clean"]).lower().split()
        return "|".join([
            str(r["part_no"]),
            t[0] if t else "",
            t[1] if len(t) > 2 else "",
            t[-1] if t else "",
            str(r["gender"]),
        ])

    df["dup_key"] = df.apply(key, axis=1)
    df["is_duplicate"] = df.duplicated("dup_key", keep="first")
    epic = df["epic_no"].where(df["epic_no"].fillna("").str.len() > 0)
    df["epic_shared"] = epic.notna() & epic.duplicated(keep=False)
    df["age_band"] = pd.cut(
        df["age"], [0, 25, 30, 35, 40, 50, 60, 70, 120],
        labels=["<=25", "26-30", "31-35", "36-40", "41-50", "51-60", "61-70", "70+"],
    )
    return df


## 4. Build the working table

`outreach_eligible` is an active voter who is not a duplicate inside the same part. That is the list worth visiting.


In [ ]:
df = finalise(raw)
df['district'] = df['header_district'].fillna('').str.upper().str.strip()
df['occupation'] = df['occupation'].fillna('').str.title().replace({'': 'Missing'})
df['occupation_group'] = df['occupation'].map(normalise_occupation)
df['part_no_num'] = pd.to_numeric(df['part_no'], errors='coerce')
df['sr_no_num'] = pd.to_numeric(df['sr_no'], errors='coerce')

df['cross_part_key'] = (
    df['district'] + '|' + df['name_clean'].str.lower() + '|' + df['gender'].astype(str)
    + '|' + df['age'].astype('Int64').astype(str)
)
df['possible_cross_part_dup'] = df.groupby('cross_part_key')['part_no'].transform('nunique') > 1
df['outreach_eligible'] = (df['status'] == 'ACTIVE') & (~df['is_duplicate'])
df['young'] = (df['age'] >= 21) & (df['age'] <= 35)

print(f'{len(df):,} rows total')
print(df['status'].value_counts().to_string())
print(f"duplicates within a part: {int(df['is_duplicate'].sum()):,}")
print(f"voters to visit: {int(df['outreach_eligible'].sum()):,}")
print(df.loc[df['outreach_eligible']].groupby('district').size().sort_values(ascending=False).to_string())


## 5. Data quality

Check these before sending teams out. A missing pincode or village makes a voter harder to find. Unclear qualifications are listed so a keyword can be added in `normalise_qualification` if one group is large.


In [ ]:
o = df[df['outreach_eligible']].copy()
checks = pd.DataFrame({
    'check': [
        'Missing age', 'Age < 21 or > 100', 'Missing gender', 'Missing qualification',
        'Unclear qualification', 'No EPIC number', 'EPIC shared by different rows',
        'Missing pincode', 'Missing village', 'Possible duplicate across parts',
    ],
    'count': [
        int(o['age'].isna().sum()),
        int(((o['age'] < 21) | (o['age'] > 100)).sum()),
        int(o['gender'].isna().sum()),
        int((o['qualification_group'] == 'Missing').sum()),
        int((o['qualification_group'] == 'Other/Unclear').sum()),
        int((~o['has_epic']).sum()),
        int(o['epic_shared'].sum()),
        int(o['pincode'].isna().sum()),
        int(o['village'].isna().sum()),
        int(o['possible_cross_part_dup'].sum()),
    ],
})
checks['% of outreach list'] = (100 * checks['count'] / max(len(o), 1)).round(1)
print(checks.to_string(index=False))
print()
print('Qualifications still unclear:')
print(o.loc[o['qualification_group'] == 'Other/Unclear', 'qualification'].value_counts().head(20).to_string())


## 6. Where a visit reaches more people

Villages and parts are ranked by how many eligible voters live there. Young voters (21–35) and teachers (B.Ed / M.Ed) are counted separately: younger graduates are the easiest new contacts, and teachers talk to many families.

Talukas with a low share of women are listed as well. Those are places where contacting women graduates adds people the roll is under-representing.


In [ ]:
import matplotlib.pyplot as plt

work = o.assign(
    is_woman=(o['gender'] == 'F').astype(int),
    is_man=(o['gender'] == 'M').astype(int),
    is_young=o['young'].fillna(False).astype(int),
    is_teacher=(o['qualification_group'] == 'Education (B.Ed/M.Ed)').astype(int),
    is_farmer=(o['occupation_group'] == 'Farmer').astype(int),
)

def place_summary(frame, keys):
    out = (frame.groupby(keys, dropna=False)
           .agg(voters=('name_clean', 'size'),
                women=('is_woman', 'sum'),
                men=('is_man', 'sum'),
                young_21_to_35=('is_young', 'sum'),
                teachers=('is_teacher', 'sum'),
                farmers=('is_farmer', 'sum'),
                parts=('part_no', 'nunique'))
           .reset_index())
    out['women_share_%'] = (100 * out['women'] / out['voters']).round(1)
    out['young_share_%'] = (100 * out['young_21_to_35'] / out['voters']).round(1)
    return out.sort_values(['voters', 'young_21_to_35'], ascending=False)

village_plan = place_summary(work, ['district', 'taluka', 'village', 'pincode'])
taluka_plan = place_summary(work, ['district', 'taluka'])
part_plan = (work.groupby(['district', 'taluka', 'part_no', 'part_no_num'], dropna=False)
             .agg(voters=('name_clean', 'size'),
                  women=('is_woman', 'sum'),
                  young_21_to_35=('is_young', 'sum'),
                  teachers=('is_teacher', 'sum'),
                  farmers=('is_farmer', 'sum'))
             .reset_index()
             .sort_values(['voters', 'district', 'part_no_num'], ascending=[False, True, True])
             .drop(columns='part_no_num'))

qual_mix = (o['qualification_group'].value_counts()
            .rename_axis('qualification_group').reset_index(name='voters'))
occ_mix = (o['occupation_group'].fillna('Missing').value_counts()
           .rename_axis('occupation_group').reset_index(name='voters'))
age_gender = (pd.crosstab(o['age_band'].astype(str), o['gender'].fillna('Missing'), dropna=False)
              .rename_axis('age_band').reset_index())
age_gender['voters'] = age_gender.drop(columns='age_band').sum(axis=1)
band_order = ['<=25', '26-30', '31-35', '36-40', '41-50', '51-60', '61-70', '70+', 'nan']
age_gender['age_band'] = pd.Categorical(age_gender['age_band'], categories=band_order, ordered=True)
age_gender = age_gender.sort_values('age_band')

shifted_cols = ['district', 'district_code', 'taluka', 'village', 'part_no', 'shifted_to',
                'name_clean', 'relative_name', 'gender', 'age', 'address', 'source_file']
shifted = df.loc[df['status'] == 'SHIFTED', shifted_cols].sort_values(
    ['district', 'shifted_to', 'name_clean'])

print(f'Voters a team can visit: {len(o):,}')
print(f'Villages: {village_plan["village"].nunique():,}    Talukas: {len(taluka_plan):,}    Parts: {len(part_plan):,}')
print(f'Young voters aged 21–35: {int(o["young"].sum()):,}')
print(f'Teachers (B.Ed / M.Ed): {int((o["qualification_group"] == "Education (B.Ed/M.Ed)").sum()):,}')
print(f'Shifted, look up in the new part: {len(shifted):,}')
print()
print('Top 20 villages — go here first')
print(village_plan.head(20).to_string(index=False))
print()
print('Talukas')
print(taluka_plan.head(15).to_string(index=False))
print()
low_women = taluka_plan[taluka_plan['voters'] >= 200].sort_values('women_share_%')
print('Talukas with the smallest share of women (at least 200 voters)')
print(low_women.head(10).to_string(index=False))
print()
print('Qualification mix')
print(qual_mix.to_string(index=False))
print()
print('Occupation mix')
print(occ_mix.head(15).to_string(index=False))

top = village_plan.head(15).iloc[::-1]
labels = top['village'].fillna('(missing village)') + ' (' + top['taluka'].fillna('') + ')'
plt.figure(figsize=(10, 6))
plt.barh(labels, top['voters'], color='#1f4e79')
plt.title('15 villages where one visit reaches the most graduates')
plt.xlabel('Eligible voters')
plt.tight_layout()
plt.show()

qplot = qual_mix.iloc[::-1]
plt.figure(figsize=(8, 4.5))
plt.barh(qplot['qualification_group'], qplot['voters'], color='#2e7d32')
plt.title('Who the voters are, by qualification')
plt.xlabel('Eligible voters')
plt.tight_layout()
plt.show()


## 7. Save the outreach workbook

`Outreach_List` is sorted so the largest villages come first. `village_voters` is how many eligible voters share that village, which is the number of people one stop can cover.


In [ ]:
how = pd.DataFrame({
    'sheet': [
        'Outreach_List', 'Village_Plan', 'Taluka_Plan', 'Part_Plan',
        'Qualification_Mix', 'Age_Gender', 'Occupation_Mix',
        'Shifted_Followup', 'Data_Quality',
    ],
    'use_it_for': [
        'Names and addresses to visit. Biggest villages are at the top.',
        'Assign a team to a village. Includes women, young voters, teachers, and farmers.',
        'Which taluka to schedule on which day.',
        'Booth / part numbers, for dividing a village among workers.',
        'Degree groups, so the message can differ for teachers, engineers, commerce, and so on.',
        'Age and gender, to see where younger voters or women are concentrated.',
        'Farmer versus other work, which changes where you meet people.',
        'Voters marked shifted. Use shifted_to instead of the old address.',
        'Missing age, pin, village, or qualification before you trust a total.',
    ],
})

visit_cols = [
    'district', 'district_code', 'taluka', 'village', 'locality', 'pincode', 'village_voters',
    'part_no', 'sr_no', 'name_clean', 'relative_name', 'gender', 'age', 'age_band', 'young',
    'qualification', 'qualification_group', 'occupation', 'occupation_group', 'epic_no', 'address',
]
sizes = o.groupby(['district', 'taluka', 'village'], dropna=False)['name_clean'].transform('size')
outreach = o.assign(village_voters=sizes).sort_values(
    ['village_voters', 'district', 'taluka', 'village', 'part_no_num', 'sr_no_num'],
    ascending=[False, True, True, True, True, True],
)
outreach['age_band'] = outreach['age_band'].astype(str)
age_gender['age_band'] = age_gender['age_band'].astype(str)

xlsx = os.path.join(WORK_DIR, 'outreach_analysis.xlsx')
with pd.ExcelWriter(xlsx, engine='openpyxl') as w:
    how.to_excel(w, sheet_name='How_To_Use', index=False)
    outreach[visit_cols].to_excel(w, sheet_name='Outreach_List', index=False)
    village_plan.to_excel(w, sheet_name='Village_Plan', index=False)
    taluka_plan.to_excel(w, sheet_name='Taluka_Plan', index=False)
    part_plan.to_excel(w, sheet_name='Part_Plan', index=False)
    qual_mix.to_excel(w, sheet_name='Qualification_Mix', index=False)
    age_gender.to_excel(w, sheet_name='Age_Gender', index=False)
    occ_mix.to_excel(w, sheet_name='Occupation_Mix', index=False)
    shifted.to_excel(w, sheet_name='Shifted_Followup', index=False)
    checks.to_excel(w, sheet_name='Data_Quality', index=False)
print('saved to', xlsx)
print(f'Outreach_List rows: {len(outreach):,}')
print(f'Shifted_Followup rows: {len(shifted):,}')
